In [1]:
# Install dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

In [4]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {"temperature": temperature},
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [5]:
import json

def generate_dataset():
    prompt = """
Genera un conjunto de datos de evaluación para evaluar *prompts*. Este conjunto de datos se utilizará para evaluar *prompts* que generen código Python, JSON o expresiones regulares (Regex) específicamente para tareas relacionadas con AWS. Genera una matriz de objetos JSON, donde cada uno represente una tarea que requiera Python, JSON o una expresión regular para completarse.

Ejemplo de salida:
```json
[
  {
    "task": "Descripción de la tarea",
    "format": "json" o "python" o "regex",
    "solution_criteria": "Criterio clave para evaluar la solución"
  },
  ...otros
]
```

* Céntrate en tareas que puedan resolverse escribiendo una única función de Python, un único objeto JSON o una única expresión regular.
* Céntrate en tareas que no requieran escribir mucho código.

Por favor generar 3 objetos.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [6]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [7]:
def run_prompt(test_case):
    """Combina el prompt y la entrada del caso de prueba, y luego devuelve el resultado."""
    prompt = f"""
Por favor resolver la siguiente tarea:

{test_case["task"]}

* Responder solamente en formato Python, JSON o una Regex plano.
* No agregar código comentado o comentarios de explicación.
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [8]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
Eres un experto en la revisión de código de AWS. Tu tarea consiste en evaluar la siguiente solución generada por IA.

Tarea Original:
<task>
{test_case["task"]}
</task>

Solución a evaluar:
<solution>
{output}
</solution>

Criterio que se debería usar para evaluar la solución:
<solution_criteria>
{test_case["solution_criteria"]}
</solution_criteria>

Formato de Salida
Proporcione su evaluación como un objeto JSON estructurado con los siguientes campos, en este orden específico:
- "strengths": Un array de 1 a 3 fortalezas clave
- "weaknesses": Un array de 1 a 3 áreas clave de mejora
- "reasoning": Una explicación concisa de su evaluación general.
- "score": Un número entre 1-10

Responde en formato JSON. Mantén la respuesta concisa y directa.
Ejemplo de estructura de respuesta:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [9]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [10]:
def run_test_case(test_case):
    """Ejecutar run_prompt luego evalúa el resultado"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [11]:
from statistics import mean

def run_eval(dataset):
    """Carga el conjunto de datos y llama a run_test_case con cada caso."""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Score Promedio: {average_score}")
    
    return results

In [12]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Score Promedio: 8.666666666666666


In [13]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\ndef extraer_servicio_arn(arn):\n    partes = arn.split(':')\n    if len(partes) >= 3:\n        return partes[2]\n    return None\n",
    "test_case": {
      "task": "Crea una funci\u00f3n de Python que reciba un ARN de AWS y extraiga el nombre del servicio (por ejemplo, de 'arn:aws:s3:::mi-bucket' deber\u00eda extraer 's3')",
      "format": "python",
      "solution_criteria": "La funci\u00f3n debe parsear correctamente el ARN y devolver el servicio AWS (tercera posici\u00f3n despu\u00e9s de 'arn:aws:')"
    },
    "score": 8.5,
    "reasoning": "La soluci\u00f3n cumple con el criterio b\u00e1sico de extraer el servicio AWS correctamente de la tercera posici\u00f3n del ARN. La l\u00f3gica es correcta y funcionar\u00e1 para ARNs v\u00e1lidos. Sin embargo, la validaci\u00f3n es m\u00ednima: solo verifica que existan 3 partes, cuando un ARN v\u00e1lido tiene al menos 6 componentes (arn:partition:service:region:account-id:resource). No valida la estructura completa